# AIaaS — MLPerf Inference Runner (Standard tier)

The **leaderboard-grade, accuracy-gated** tier. This notebook drives the official
**MLPerf Inference reference benchmark** — LoadGen + the reference app — for the
**vision classification & detection** workloads (resnet50, retinanet, mobilenet),
modeled on the `vision/classification_and_detection/GettingStarted.ipynb` tutorial
in the **`kurtvalcorza/inference`** fork.

Unlike the proxy notebooks, this measures what MLPerf measures:
- a **LoadGen scenario** (Offline / Server / SingleStream) — real traffic, not a
  batch sweep
- an **accuracy gate** (Top-1 for resnet50, mAP for retinanet)
- the official **`mlperf_log_summary.txt`** (QPS / latency percentiles / VALID)

### Honest scope
This is **heavy**: it clones + builds the fork (LoadGen C++ + the app), downloads a
model, and needs a dataset. The defaults run a **smoke test** (mobilenet on a
*fake* imagenet) so the pipeline works end-to-end in minutes; switch `MODEL` +
point `DATA_DIR` at the real **ImageNet** (resnet50) or **OpenImages** (retinanet)
for a credible run. For a fully reproducible, submission-grade path use the
**MLCFlow** automation (`mlcr run-mlperf-inference-app ...`) — see the notes at the
end. The actual runs belong in a session scoped to the `inference` fork; this
notebook is the portable bridge.

## 1. Clone the fork + build LoadGen and the app
References your `kurtvalcorza/inference` fork (the Standard-tier home).

In [ ]:
import os, subprocess, sys, time, json

REPO_URL = "https://github.com/kurtvalcorza/inference"
INF = os.path.abspath("inference-ref")
APP = os.path.join(INF, "vision", "classification_and_detection")

def sh(cmd, cwd=None, env=None):
    print("$", " ".join(cmd), f"(cwd={cwd})" if cwd else "")
    p = subprocess.run(cmd, cwd=cwd, env=env, capture_output=True, text=True)
    if p.stdout: print(p.stdout[-1500:])
    if p.returncode != 0:
        print("STDERR:", p.stderr[-1500:])
    return p

if not os.path.isdir(INF):
    sh(["git", "clone", "--recurse-submodules", "--depth", "1", REPO_URL, INF])

# Build + install LoadGen (C++ + pybind11 bindings) and the reference app.
# Use `pip install <dir>` rather than `setup.py develop`: develop registers the
# module via easy-install.pth, which the *already-running* kernel won't re-read,
# so `import mlperf_loadgen` fails mid-session. pip installs into site-packages —
# importable immediately (and `setup.py develop` is deprecated in modern setuptools).
sh([sys.executable, "-m", "pip", "install", "-q", "pybind11"])
_lg = sh([sys.executable, "-m", "pip", "install", os.path.join(INF, "loadgen")],
         env={**os.environ, "CFLAGS": "-std=c++14"})
if _lg.returncode != 0:
    raise RuntimeError("LoadGen build/install failed (see STDERR above) — fix build deps and re-run.")
_app = sh([sys.executable, "-m", "pip", "install", APP])
if _app.returncode != 0:
    raise RuntimeError("Reference app build/install failed (see STDERR above).")
import mlperf_loadgen   # hard import: stop here if the build didn't produce the module
print("mlperf_loadgen OK ->", mlperf_loadgen.__file__)

## 2. Backend dependencies
ONNX Runtime backend (works everywhere; use `onnxruntime-gpu` on a GPU). PyTorch /
TensorFlow backends also work — see the app README.

In [ ]:
import torch
USE_GPU = torch.cuda.is_available()
ort_pkg = "onnxruntime-gpu" if USE_GPU else "onnxruntime"
sh([sys.executable, "-m", "pip", "install", "-q", ort_pkg,
    "pycocotools", "opencv-python-headless", "numpy", "pillow", "pandas"])
print("backend:", ort_pkg, "| GPU:", USE_GPU)


## 3. Environment & hardware capture

In [ ]:
import platform, datetime

def smi(field):
    try:
        out = subprocess.run(["nvidia-smi", f"--query-gpu={field}",
                              "--format=csv,noheader,nounits"],
                             capture_output=True, text=True, timeout=10)
        return [x.strip() for x in out.stdout.strip().splitlines() if x.strip()]
    except Exception:
        return []

def detect_platform():
    if "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ: return "colab"
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ: return "kaggle"
    if os.path.exists("/opt/.sagemakerinternal"): return "sagemaker-studio-lab"
    return "local/on-prem"

ENV = {
    "platform": detect_platform(),
    "timestamp": datetime.datetime.utcnow().isoformat() + "Z",
    "gpu_name": (torch.cuda.get_device_name(0) if USE_GPU else "cpu"),
    "vram_total_gb": (round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
                      if USE_GPU else None),
    "cuda": torch.version.cuda, "driver": (smi("driver_version") or ["?"])[0],
    "torch": torch.__version__, "python": platform.python_version(),
}
print(json.dumps(ENV, indent=2))


## 4. Configuration
Defaults = **smoke test** (mobilenet + fake imagenet). For credible runs, switch
`MODEL` and point `DATA_DIR` at the real dataset (see notes).

In [ ]:
# --- choose the benchmark ---
MODEL    = "mobilenet"          # smoke. Real: "resnet50" (ImageNet) | "retinanet" (OpenImages)
BACKEND  = "onnxruntime"
DEVICE   = "gpu" if USE_GPU else "cpu"
SCENARIO = "Offline"            # Offline | Server | SingleStream | MultiStream
ACCURACY = True                 # also run the accuracy pass (gate)

# Model download URLs (MLPerf reference models)
MODEL_URLS = {
    "mobilenet": "https://zenodo.org/record/3157894/files/mobilenet_v1_1.0_224.onnx",
    "resnet50":  "https://zenodo.org/record/4735647/files/resnet50_v1.onnx",
    "retinanet": "https://zenodo.org/record/6617879/files/resnext50_32x4d_fpn.onnx",
}

# Smoke run keeps it short; remove EXTRA_OPS for a full/valid run.
EXTRA_OPS = "--time 10 --max-latency 0.2" if MODEL == "mobilenet" else ""

# Dataset: 'fake' builds a tiny stand-in imagenet (smoke only). 'real' uses DATA_DIR.
DATASET_MODE = "fake" if MODEL == "mobilenet" else "real"
DATA_DIR = ""   # set to your ImageNet val / OpenImages dir for a real run

OUT_DIR = os.path.abspath("mlperf_results")
os.makedirs(OUT_DIR, exist_ok=True)
print(f"MODEL={MODEL}, BACKEND={BACKEND}, DEVICE={DEVICE}, SCENARIO={SCENARIO}, "
      f"dataset={DATASET_MODE}")

# --- shared LoadGen-summary helpers (used by BOTH the reference-app path and the
#     MLCFlow suite path, so they live here in the common config cell) ---
import re

def parse_summary(path):
    """Parse a LoadGen mlperf_log_summary.txt into {label: value}."""
    out = {}
    if path and os.path.exists(path):
        with open(path) as fh:
            for line in fh:
                m = re.match(r"\s*(.+?)\s*:\s*(.+?)\s*$", line)
                if m:
                    k, v = m.group(1).strip(), m.group(2).strip()
                    if re.match(r"^[-+]?\d*\.?\d+([eE][-+]?\d+)?$", v):
                        try:
                            v = float(v)
                        except ValueError:
                            pass
                    out[k] = v
    return out

def pick(d, *names):
    """First value whose key contains one of `names`, names tried in order."""
    for n in names:
        for k, v in d.items():
            if n.lower() in k.lower():
                return v
    return None


## 5. Get the model and dataset

In [ ]:
# model
model_file = os.path.join(APP, os.path.basename(MODEL_URLS[MODEL]))
if not os.path.exists(model_file) or os.path.getsize(model_file) == 0:
    if os.path.exists(model_file):
        os.remove(model_file)   # drop a 0-byte file from a previous failed download
    r = sh(["wget", "-q", MODEL_URLS[MODEL], "-O", model_file])
    if r.returncode != 0 or not os.path.exists(model_file) or os.path.getsize(model_file) == 0:
        if os.path.exists(model_file):
            os.remove(model_file)
        raise RuntimeError(f"model download failed or empty: {MODEL_URLS[MODEL]}")
print("model:", model_file, os.path.getsize(model_file), "bytes")

# dataset
import glob
if DATASET_MODE == "fake":
    # Self-contained synthetic fake-ImageNet for the smoke path. The fork's
    # tools/make_fake_imagenet.sh downloads 8 images from Wikimedia via wget, which
    # routinely 403s on cloud runtimes (User-Agent policy) and leaves an empty val/
    # dir -> "no images in image list found". Generating locally with PIL removes
    # that network dependency and keeps images + val_map.txt consistent.
    import shutil
    import numpy as np
    from PIL import Image
    DATA_DIR = os.path.join(APP, "fake_imagenet")
    if os.path.isdir(DATA_DIR):
        shutil.rmtree(DATA_DIR)   # clear any partial set from a prior run
    val = os.path.join(DATA_DIR, "val")
    os.makedirs(val, exist_ok=True)
    with open(os.path.join(DATA_DIR, "val_map.txt"), "w") as _vm:
        for i in range(8):
            arr = (np.random.rand(224, 224, 3) * 255).astype("uint8")
            fn = f"img{i}.jpg"
            Image.fromarray(arr).save(os.path.join(val, fn), "JPEG")
            print(f"val/{fn} {i}", file=_vm)   # "<relative_path> <class_id>"
    imgs = [f for f in glob.glob(os.path.join(DATA_DIR, "**", "*.*"), recursive=True)
            if f.lower().endswith((".jpeg", ".jpg", ".png"))]
    if not imgs:
        raise RuntimeError("failed to generate synthetic fake_imagenet images.")
    print(f"fake_imagenet: {len(imgs)} synthetic image(s)")
else:
    assert DATA_DIR and os.path.isdir(DATA_DIR), (
        "Set DATA_DIR to a real dataset (ImageNet val for resnet50, "
        "OpenImages for retinanet). See the app README for download scripts.")
print("DATA_DIR:", DATA_DIR)

## 6. Run the MLPerf reference benchmark
`run_local.sh <backend> <model> <device> --scenario <S> [--accuracy]`, driven by
LoadGen. Official logs land in `mlperf_log_summary.txt` / `mlperf_log_detail.txt`.

In [ ]:
run_env = {**os.environ, "MODEL_DIR": APP, "DATA_DIR": DATA_DIR, "EXTRA_OPS": EXTRA_OPS}
import glob

def run_local(extra=()):
    cmd = ["bash", "run_local.sh", BACKEND, MODEL, DEVICE, "--scenario", SCENARIO, *extra]
    t0 = time.time()
    p = sh(cmd, cwd=APP, env=run_env)
    return p, round(time.time() - t0, 1)

def newest(pattern, root):
    cands = sorted(glob.glob(os.path.join(root, "**", pattern), recursive=True),
                   key=os.path.getmtime)
    return cands[-1] if cands else None

# parse_summary() / pick() are defined in the Configuration cell (shared helpers).

# 1) PERFORMANCE run — the mode that writes QPS / latency / "Result is VALID".
#    (--accuracy runs LoadGen in AccuracyOnly mode, which emits NO performance summary,
#     so it must be a separate run; running only --accuracy gives an all-None result.)
perf_proc, elapsed = run_local()
print(f"\nperformance run rc={perf_proc.returncode} in {elapsed}s")
run_out = os.path.join(APP, "output", f"{MODEL}-{BACKEND}-{DEVICE}")

# Gate on the perf run and scope strictly to THIS run's output dir — no app-wide
# fallback, so a failed run can't resurrect a previous model's summary. Parse the
# summary CONTENTS now, before the accuracy run overwrites mlperf_log_summary.txt.
SUMMARY, RESULTS = {}, {}
if perf_proc.returncode != 0:
    print("WARNING: performance run failed (rc != 0) — no QPS/VALID summary recorded.")
else:
    summary_txt = newest("mlperf_log_summary.txt", run_out)
    results_json = newest("results.json", run_out)
    SUMMARY = parse_summary(summary_txt)
    if results_json and os.path.exists(results_json):
        try:
            with open(results_json) as fh:
                RESULTS = json.load(fh)
        except Exception as e:
            print("results.json parse error:", e)
    print("perf summary:", summary_txt)
    print("results.json:", results_json)

# 2) ACCURACY run — separate mode; produces the accuracy log, not a perf summary.
#    Only run it if the perf pass succeeded (otherwise there's nothing to gate).
ACC_LOG = None
if ACCURACY and perf_proc.returncode == 0:
    acc_proc, acc_elapsed = run_local(["--accuracy"])
    print(f"accuracy run rc={acc_proc.returncode} in {acc_elapsed}s")
    if acc_proc.returncode == 0:
        ACC_LOG = newest("mlperf_log_accuracy.json", run_out) or newest("accuracy.txt", run_out)
    else:
        print("WARNING: accuracy run failed — accuracy gate not evaluated.")


## 7. Parse + normalize results

In [ ]:
# SUMMARY / RESULTS were parsed in the previous cell (from the PERFORMANCE run,
# before the accuracy pass overwrote the summary file).
NORM = {
    "schema": "mlperf-inference/1.0",
    "env": ENV, "model": MODEL, "backend": BACKEND, "device": DEVICE,
    "scenario": SCENARIO,
    "performance_ok": perf_proc.returncode == 0,
    "accuracy_ran": ACC_LOG is not None,
    "dataset_mode": DATASET_MODE, "elapsed_s": elapsed,
    "accuracy_log": ACC_LOG,
    "loadgen_summary": SUMMARY, "app_results": RESULTS,
}
tag = (ENV["platform"] + "_" + ENV["gpu_name"] + "_" + MODEL + "_" + SCENARIO).replace(" ", "-").replace("/", "-")
out = os.path.join(OUT_DIR, f"mlperf_{tag}.json")
with open(out, "w") as f:
    json.dump(NORM, f, indent=2)
print("Saved:", out)

print("\n==== MLPERF SUMMARY ====")
print(f"{ENV['gpu_name']} | {MODEL} | {SCENARIO} | {BACKEND}")
print("Result valid :", pick(SUMMARY, "Result is", "Result:"))
# Completed (achieved) before Scheduled (target) — in Server scenario LoadGen emits
# both, and 'Samples per second' is a substring of 'Scheduled samples per second'.
print("QPS/throughput:", pick(SUMMARY, "Completed samples per second", "Samples per second",
                              "Scheduled samples per second", "QPS"))
print("Mean latency  :", pick(SUMMARY, "Mean latency"))
print("90th pct ns   :", pick(SUMMARY, "90.00 percentile", "90th percentile"))
print("Accuracy      :", pick(RESULTS, "accuracy", "mAP") if RESULTS else "(see accuracy log)")


## 8. Notes — real runs, datasets, and the MLCFlow path

- **Datasets:** resnet50 → **ImageNet** `val` (manual download/registration);
  retinanet → **OpenImages** (the app has download scripts under `tools/`). Set
  `DATASET_MODE="real"` and `DATA_DIR` accordingly, and drop `EXTRA_OPS` so the run
  meets MLPerf's minimum duration/queries for a **VALID** result.
- **Scenarios:** `Offline` (max throughput) and `Server` (Poisson arrivals under a
  p99 bound) are the datacenter scenarios; `SingleStream`/`MultiStream` are edge.
- **Submission-grade / reproducible:** prefer **MLCFlow** —
  `pip install mlc-scripts` then `mlcr run-mlperf-inference-app --model=retinanet
  --scenario=Offline --device=cuda ...` — which handles models, datasets, accuracy,
  and the submission tree. See `SubmissionExample.ipynb` in the fork.
- **LLM Standard tier:** the same MLPerf suite has `llama3.1-8b` and `whisper`
  (v5.1+) — run those in the fork to get the accuracy-gated LLM/ASR numbers that
  complement our Comparable-tier vLLM notebook.
- This notebook is the portable bridge; the heavy/credible runs belong in an
  `inference`-scoped session.

## 9. The MLPerf Inference **suite** via MLCFlow

The vision flow above uses the fork's `run_local.sh`. For the rest of the MLPerf
Inference **suite**, use **MLCFlow / CM** — it fetches each model + dataset, runs
LoadGen, and applies the accuracy gate. This section takes a **list of models** and
runs them in a loop, so you get the suite in one pass.

**Datacenter suite** (set `MLC_MODELS` to the subset you want):
`resnet50`, `retinanet`, `3d-unet-99(.9)`, `bert-99(.9)`, `dlrm-v2-99(.9)`,
`gptj-99(.9)`, `sdxl`, `llama2-70b-99(.9)`, `mixtral-8x7b`, `llama3.1-8b`,
`whisper`, `deepseek-r1`. The default list is a **single-GPU-fittable subset**; large
models (llama2-70b, mixtral, deepseek-r1) need more VRAM / multi-GPU.
`execution_mode=test` is a smoke run; use `valid` for submission-grade.

In [ ]:
import subprocess, sys, os, json, shutil

# MLCFlow (mlc / mlcr) is the CURRENT MLCommons automation interface, replacing the
# older CM (cm / cmr) + cm4mlops path — `cm pull repo mlcommons@cm4mlops` no longer
# registers the run-mlperf scripts, which surfaces as "no scripts were found with
# above tags". `mlc-scripts` bundles the run-mlperf automation, so `mlcr` finds it
# with no separate pull step.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mlc-scripts"], check=False)
print("mlcr on PATH:", shutil.which("mlcr"), "| mlc:", shutil.which("mlc"))

In [ ]:
# MLPerf Inference SUITE via MLCFlow — set the models; the runner loops over them.
MLC_MODELS     = ["resnet50", "retinanet", "sdxl", "whisper"]   # single-GPU-fittable subset; edit freely
MLC_SCENARIO   = "Offline"
MLC_DEVICE     = "cuda" if USE_GPU else "cpu"
MLC_IMPL       = "mlcommons-python"     # reference implementation (default)
MLC_EXEC_MODE  = "test"                 # "test" (smoke) | "valid" (submission-grade)
MLC_CATEGORY   = "datacenter"           # "datacenter" | "edge"
MLC_RESULTS    = os.path.abspath("mlperf_mlc_results")
os.makedirs(MLC_RESULTS, exist_ok=True)

# Per-model implementation/framework flags. The generic reference path is
# mlcommons-python + onnxruntime (resnet50, retinanet); some models need a
# different framework: sdxl -> pytorch, whisper -> reference + vllm (per the
# MLCommons docs). Adjust per your mlc-scripts version if a flag is rejected.
DEFAULT_FLAGS = [f"--implementation={MLC_IMPL}", "--framework=onnxruntime"]
MODEL_FLAGS = {
    "sdxl":    ["--implementation=mlcommons-python", "--framework=pytorch"],
    "whisper": ["--implementation=reference", "--framework=vllm"],
}

import time, shutil
# MLCFlow's runner CLI: prefer `mlcr` (current MLCFlow); fall back to `cmr` (legacy CM).
MLC_CLI = shutil.which("mlcr") or shutil.which("cmr")

SUITE = {}
for model in MLC_MODELS:
    if MLC_CLI is None:
        print(f"  skip {model}: no MLCFlow CLI (mlcr/cmr) on PATH — run the install cell / restart")
        SUITE[model] = {"returncode": None, "elapsed_s": None, "error": "no mlcflow cli"}
        continue
    cmd = [
        MLC_CLI, "run-mlperf,inference",
        f"--model={model}", f"--scenario={MLC_SCENARIO}", f"--device={MLC_DEVICE}",
        *MODEL_FLAGS.get(model, DEFAULT_FLAGS),
        f"--execution_mode={MLC_EXEC_MODE}",
        f"--category={MLC_CATEGORY}", f"--results_dir={MLC_RESULTS}", "--quiet=true",
    ]
    print("\n=== MLPerf:", model, "===\n$", " ".join(cmd))
    t0 = time.time()
    try:
        p = subprocess.run(cmd, capture_output=True, text=True)
        print(p.stdout[-1200:])
        if p.returncode != 0:
            print("STDERR:", p.stderr[-1200:])
        SUITE[model] = {"returncode": p.returncode, "elapsed_s": round(time.time() - t0, 1)}
    except Exception as e:
        print(f"  {model}: MLCFlow run not launchable: {type(e).__name__}: {e}")
        SUITE[model] = {"returncode": None, "elapsed_s": round(time.time() - t0, 1), "error": str(e)}

In [ ]:
import glob
import pandas as pd

# Map each suite model to the path tokens MLCFlow may use for its results dir —
# the CLI alias ("sdxl") differs from the on-disk name ("stable-diffusion-xl").
MODEL_DIR_TOKENS = {
    "sdxl":    ["sdxl", "stable-diffusion-xl", "stable_diffusion"],
    "whisper": ["whisper", "speech_to_text", "speech-to-text"],
}

rows = []
for model in MLC_MODELS:
    tokens = MODEL_DIR_TOKENS.get(model, [model.split("-")[0]])
    # Match tokens against the path RELATIVE to MLC_RESULTS (so a token in a parent
    # dir / cwd can't cross-attribute). No global fallback: a failed model must not
    # inherit another model's VALID/QPS.
    cands = sorted(
        (f for f in glob.glob(os.path.join(MLC_RESULTS, "**", "mlperf_log_summary.txt"), recursive=True)
         if any(t in os.path.relpath(f, MLC_RESULTS).lower() for t in tokens)),
        key=os.path.getmtime)
    summ = parse_summary(cands[-1]) if cands else {}
    info = SUITE.get(model, {"returncode": None, "elapsed_s": None})
    NORM = {
        "schema": "mlperf-inference/1.0", "via": "mlcflow", "env": ENV, "model": model,
        "scenario": MLC_SCENARIO, "device": MLC_DEVICE, "implementation": MLC_IMPL,
        "execution_mode": MLC_EXEC_MODE, "category": MLC_CATEGORY,
        "returncode": info.get("returncode"), "elapsed_s": info.get("elapsed_s"),
        "loadgen_summary": summ,
    }
    with open(os.path.join(OUT_DIR, f"mlperf_mlc_{model}_{MLC_SCENARIO}.json"), "w") as f:
        json.dump(NORM, f, indent=2)
    rows.append({"model": model, "rc": info.get("returncode"),
                 "valid": pick(summ, "Result is", "Result:"),
                 "qps": pick(summ, "Completed samples per second", "Samples per second",
                             "Scheduled samples per second"),
                 "elapsed_s": info.get("elapsed_s")})

print("\n==== MLPERF INFERENCE SUITE ====")
df = pd.DataFrame(rows)
try:
    from IPython.display import display; display(df)
except Exception:
    print(df.to_string(index=False))
print("test mode = smoke; execution_mode=valid for VALID numbers. Per-model JSON in", OUT_DIR)
